## 4. モデルを比較する
### 4.7 モデル選択の例
#### 図4.1
$$ \text{MORT} = \beta_0 + \beta_1 \text{EDUC} + \beta_2 \text{NONW} + \beta_3 \text{LSO2} + \epsilon $$

In [1]:
df <- read.csv('./src/airpollution.csv')
df$LNOX <- log10(df$NOX)
df$LSO2 <- log10(df$SO2)
head(df)

,CITY,RAIN,EDUC,POPD,NONW,NOX,SO2,MORT,LNOX,LSO2
,<chr>,<int>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<dbl>,<dbl>
1,akronOH,36,11.4,3243,8.8,15,59,921.9,1.1760913,1.770852
2,albanyNY,35,11.0,4281,3.5,10,39,997.9,1.0000000,1.591065
3,allenPA,44,9.8,4260,0.8,6,33,962.4,0.7781513,1.518514
4,atlantGA,47,11.1,3125,27.1,8,24,982.3,0.9030900,1.380211
5,baltimMD,43,9.6,6441,24.4,38,206,1071.0,1.5797836,2.313867
6,birmhmAL,53,10.2,3325,38.5,32,72,1030.0,1.5051500,1.857332


In [2]:
res <- lm(MORT ~ EDUC + NONW + LSO2, data = df)
summary(res)


Call:
lm(formula = MORT ~ EDUC + NONW + LSO2, data = df)

Residuals:
    Min      1Q  Median      3Q     Max 
-98.895 -20.870   1.628  22.745 105.138 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) 1111.9379    73.5464  15.119  < 2e-16 ***
EDUC         -23.4120     6.2173  -3.766 0.000401 ***
NONW           3.9178     0.5702   6.871 5.66e-09 ***
LSO2          27.9767     7.9132   3.535 0.000825 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 38.21 on 56 degrees of freedom
Multiple R-squared:  0.6419,	Adjusted R-squared:  0.6227 
F-statistic: 33.46 on 3 and 56 DF,  p-value: 1.594e-12


Rの`AIC(res)`だと613になる

In [3]:
n <- nrow(df)
p <- res$rank - 1 # rankは切片も含む

SS_R <- sum((res$fitted.values - mean(df$MORT))^2)
SS_E <- sum((df$MORT - res$fitted.values)^2)
SS <- sum((df$MORT - mean(df$MORT))^2)

V_R <- SS_R / p
V_E <- SS_E / (n-p-1)
V <- SS / (n-1)

RSS <- SS_E

F <- V_R / V_E
Rsq <- SS_R / SS
Radjsq <- 1 - V_E / V
AIC <- n*log(RSS/n) + 2*(p+1)

cat('F=', F, ', R^2=', Rsq, ', R*^2=', Radjsq, ', AIC=', AIC, sep = '')

F=33.45737, R^2=0.6418799, R*^2=0.6226949, AIC=441.0449

In [4]:
library(olsrr)

full_res <- lm(MORT ~ RAIN + EDUC + POPD + NONW + LSO2 + LNOX, data = df)
cat('C_p=', ols_mallows_cp(res, full_res), sep = '')


Attaching package: ‘olsrr’


The following object is masked from ‘package:datasets’:

    rivers




C_p=9.474494

In [5]:
allen_cv <- function(res){return(sum((res$residuals / (1 - hatvalues(res)))^2) / length(res$residuals))}

cat('s_CV^2=', allen_cv(res), sep = '')

s_CV^2=1685.458

In [6]:
length(res$residuals)

[1] 60

### 問題
#### 問題4.3
AllenのCV規準

In [7]:
full_covariates <- c('RAIN', 'EDUC', 'POPD', 'NONW', 'LNOX', 'LSO2')
obj_var <- 'MORT'

list_s_CV_q = c()

for (q in 1:length(full_covariates)){
  # サイズqのサブモデルでs_CV^2を計算する
  criteria_size_q <- c() # サイズqの全サブモデルの規準を格納
  reduced_covariates_list <- combn(full_covariates, q) # サイズqの全サブモデル
  for (j in 1:ncol(reduced_covariates_list)){
    reduced_covariates <- reduced_covariates_list[, j] # モデルに使用する共変量
    df_est_j <- df[c(reduced_covariates, obj_var)]
    names(df_est_j)[names(df_est_j) == obj_var] <- "obj_var"
    res <- lm(formula = obj_var ~ ., data = df_est_j)
    criteria_size_q <- c(criteria_size_q, allen_cv(res))
  }
  list_s_CV_q = c(list_s_CV_q, min(criteria_size_q))
}

rbind(c(1:length(full_covariates)), round(list_s_CV_q))

1,2,3,4,5,6
2393,1876,1618,1576,1726,1964


HjorthのCMV規準（再現できず．．．）

In [8]:
full_covariates <- c('RAIN', 'EDUC', 'POPD', 'NONW', 'LNOX', 'LSO2')
obj_var <- 'MORT'

list_s_CMV_q = c()

# for (q in 3:3){
for (q in 1:length(full_covariates)){
  # サイズqのサブモデルでs_CMV^2を計算する
  list_y_i_q = c()
  for (i in 1:nrow(df)) {
    df_est_i <- df[-i, ] # D_{EST, i} i組目を除いた(n-1)組のデータ
    criteria_size_q <- c() # サイズqの全サブモデルの規準を格納
    reduced_covariates_list <- combn(full_covariates, q) # サイズqの全サブモデル
    for (j in 1:ncol(reduced_covariates_list)){
      reduced_covariates <- reduced_covariates_list[, j] # モデルに使用する共変量
      df_est_i_j <- df_est_i[c(reduced_covariates, obj_var)]
      names(df_est_i_j)[names(df_est_i_j) == obj_var] <- "obj_var"
      res <- lm(formula = obj_var ~ ., data = df_est_i_j)
      criteria_size_q <- c(criteria_size_q, allen_cv(res))
    }

    j_best_model_size_q <- which.min(criteria_size_q) # 最適モデル M(q, D_{EST, i})
    reduced_covariates_best_model_size_q <- reduced_covariates_list[, j_best_model_size_q]

    # print(reduced_covariates_best_model_size_q)

    df_est_i_j_best_model_size_q <- df_est_i[c(reduced_covariates_best_model_size_q, obj_var)]
    names(df_est_i_j_best_model_size_q)[names(df_est_i_j_best_model_size_q) == obj_var] <- "obj_var"
    res_best_model_size_q <- lm(formula = obj_var ~ ., data = df_est_i_j_best_model_size_q)
    y_i_q <- predict(res_best_model_size_q, newdata = df)[i] # \hat{y}_i(q)
    list_y_i_q = c(list_y_i_q, y_i_q)
  }

  s_CMV_q = sum((df[obj_var] - list_y_i_q)^2) / nrow(df)
  list_s_CMV_q = c(list_s_CMV_q, s_CMV_q)
}

rbind(c(1:length(full_covariates)), round(list_s_CMV_q))

1,2,3,4,5,6
2393,2458,2033,2034,2169,1964


`q=3`では`i=59`で`[1] "EDUC" "POPD" "NONW"`が最適モデルになっている．他の`i`では`[1] "RAIN" "NONW" "LSO2"`が最適モデルを与える．

In [9]:
df_est_i <- df[-59, ]

allen_cv(lm(MORT ~ RAIN + NONW + LSO2, data = df_est_i))
allen_cv(lm(MORT ~ EDUC + NONW + POPD, data = df_est_i))

[1] 1634.597

[1] 1528.113

選択基準に$R^2_\text{adj}$を使用した場合

In [10]:
full_covariates <- c('RAIN', 'EDUC', 'POPD', 'NONW', 'LNOX', 'LSO2')
obj_var <- 'MORT'

list_s_CMV_q = c()

# for (q in 3:3){
for (q in 1:length(full_covariates)){
  # サイズqのサブモデルでs_CMV^2を計算する
  list_y_i_q = c()
  for (i in 1:nrow(df)) {
    df_est_i <- df[-i, ] # D_{EST, i} i組目を除いた(n-1)組のデータ
    criteria_size_q <- c() # サイズqの全サブモデルの規準を格納
    reduced_covariates_list <- combn(full_covariates, q) # サイズqの全サブモデル
    for (j in 1:ncol(reduced_covariates_list)){
      reduced_covariates <- reduced_covariates_list[, j] # モデルに使用する共変量
      df_est_i_j <- df_est_i[c(reduced_covariates, obj_var)]
      names(df_est_i_j)[names(df_est_i_j) == obj_var] <- "obj_var"
      res <- lm(formula = obj_var ~ ., data = df_est_i_j)
      criteria_size_q <- c(criteria_size_q, summary(res)$adj.r.squared) # R^2adjを選択基準としてみる
    }

    j_best_model_size_q <- which.max(criteria_size_q) # 最適モデル M(q, D_{EST, i})
    reduced_covariates_best_model_size_q <- reduced_covariates_list[, j_best_model_size_q]

    # print(reduced_covariates_best_model_size_q)

    df_est_i_j_best_model_size_q <- df_est_i[c(reduced_covariates_best_model_size_q, obj_var)]
    names(df_est_i_j_best_model_size_q)[names(df_est_i_j_best_model_size_q) == obj_var] <- "obj_var"
    res_best_model_size_q <- lm(formula = obj_var ~ ., data = df_est_i_j_best_model_size_q)
    y_i_q <- predict(res_best_model_size_q, newdata = df)[i] # \hat{y}_i(q)
    list_y_i_q = c(list_y_i_q, y_i_q)
  }

  s_CMV_q = sum((df[obj_var] - list_y_i_q)^2) / nrow(df)
  list_s_CMV_q = c(list_s_CMV_q, s_CMV_q)
}

rbind(c(1:length(full_covariates)), round(list_s_CMV_q))

1,2,3,4,5,6
2393,2799,1618,1905,2054,1964
